1. Read the Netflix dataset

In [0]:
df = spark.table("workspace.default.netflix_titles")

In [0]:
display(df.limit(20))

show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,null,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable."
s2,TV Show,Blood & Water,null,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth."
s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noureddine Farihi, Geert Van Rampelberg, Bakary Diombera",null,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adventure","To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war."
s4,TV Show,Jailbirds New Orleans,null,null,null,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down among the incarcerated women at the Orleans Justice Center in New Orleans on this gritty reality series."
s5,TV Show,Kota Factory,null,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam Khan, Ahsaas Channa, Revathi Pillai, Urvi Singh, Arun Kumar",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV Comedies","In a city of coaching centers known to train India’s finest collegiate minds, an earnest but unexceptional student and his friends navigate campus life."
s6,TV Show,Midnight Mass,Mike Flanagan,"Kate Siegel, Zach Gilford, Hamish Linklater, Henry Thomas, Kristin Lehman, Samantha Sloyan, Igby Rigney, Rahul Kohli, Annarah Cymone, Annabeth Gish, Alex Essoe, Rahul Abburi, Matt Biedel, Michael Trucco, Crystal Balint, Louis Oliver",null,"September 24, 2021",2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries","The arrival of a charismatic young priest brings glorious miracles, ominous mysteries and renewed religious fervor to a dying town desperate to believe."
s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, Sofia Carson, Liza Koshy, Ken Jeong, Elizabeth Perkins, Jane Krakowski, Michael McKean, Phil LaMarr",null,"September 24, 2021",2021,PG,91 min,Children & Family Movies,"Equestria's divided. But a bright-eyed hero believes Earth Ponies, Pegasi and Unicorns should be pals — and, hoof to heart, she’s determined to prove it."
s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra Duah, Nick Medley, Mutabaruka, Afemo Omilami, Reggie Carter, Mzuri","United States, Ghana, Burkina Faso, United Kingdom, Germany, Ethiopia","September 24, 2021",1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model slips back in time, becomes enslaved on a plantation and bears witness to the agony of her ancestral past."
s9,TV Show,The Great British Baking Show,Andy Devonshire,"Mel Giedroyc, Sue Perkins, Mary Berry, Paul Hollywood",United Kingdom,"September 24, 2021",2021,TV-14,9 Seasons,"British TV Shows, Reality TV","A talented batch of amateur bakers face off in a 10-week competition, whipping up their best dishes in the hopes of being named the U.K.'s best."
s10,Movie,The Starling,Theodore Melfi,"Melissa McCarthy, Chris O'Dowd, Kevin Kline, Timothy Olyphant, Daveed Diggs, Skyler Gisondo, Laura Harrier, Rosalind Chao, 

2. Count Movies vs TV Shows

In [0]:
display(
    df.groupBy("type")
      .count()
)

type,count
Movie,6131
TV Show,2676
null,1
William Wyler,1


3. Count missing director values

In [0]:
df.filter(df["director"].isNull()).count()


2636

4. Count of titles per rating, sorted descending

In [0]:
df.groupBy("rating") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+--------------------+-----+
|              rating|count|
+--------------------+-----+
|               TV-MA| 3207|
|               TV-14| 2160|
|               TV-PG|  862|
|                   R|  799|
|               PG-13|  490|
|               TV-Y7|  334|
|                TV-Y|  307|
|                  PG|  287|
|                TV-G|  220|
|                  NR|   80|
|                   G|   41|
|            TV-Y7-FV|    6|
|                NULL|    6|
|               NC-17|    3|
|                  UR|    3|
|              74 min|    1|
|              84 min|    1|
|              66 min|    1|
|Classic Movies, D...|    1|
+--------------------+-----+



5. Top 10 listed_in genres by title count

In [0]:
df.groupBy("listed_in") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(10)

+--------------------+-----+
|           listed_in|count|
+--------------------+-----+
|Dramas, Internati...|  362|
|       Documentaries|  359|
|     Stand-Up Comedy|  334|
|Comedies, Dramas,...|  274|
|Dramas, Independe...|  252|
|            Kids' TV|  220|
|Children & Family...|  215|
|Children & Family...|  201|
|Documentaries, In...|  186|
|Dramas, Internati...|  180|
+--------------------+-----+
only showing top 10 rows


6. Top 15 genres by title count

In [0]:
from pyspark.sql.functions import split, explode, trim, col
df \
    .withColumn("genre", explode(split(col("listed_in"), ","))) \
    .withColumn("genre", trim(col("genre"))) \
    .groupBy("genre") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(15, truncate=False)

+------------------------+-----+
|genre                   |count|
+------------------------+-----+
|International Movies    |2752 |
|Dramas                  |2427 |
|Comedies                |1674 |
|International TV Shows  |1351 |
|Documentaries           |868  |
|Action & Adventure      |859  |
|TV Dramas               |763  |
|Independent Movies      |756  |
|Children & Family Movies|641  |
|Romantic Movies         |616  |
|TV Comedies             |581  |
|Thrillers               |577  |
|Crime TV Shows          |470  |
|Kids' TV                |451  |
|Docuseries              |395  |
+------------------------+-----+
only showing top 15 rows


7. Type of available content

In [0]:
from pyspark.sql.functions import col

df.filter(
    col("type").isin("Movie", "TV Show")
).select("type").distinct().show()

+-------+
|   type|
+-------+
|  Movie|
|TV Show|
+-------+



8.Quality of newly added contents by month and year

In [0]:
from pyspark.sql.functions import col, try_to_date, year, month, count
df = df.withColumn(
    "added_date",
    try_to_date(trim(col("date_added")), "MMMM d, yyyy")
)
df = df.withColumn("year", year(col("added_date"))) \
       .withColumn("month", month(col("added_date")))
df.filter(col("added_date").isNotNull()) \
  .groupBy("year", "month") \
  .count() \
  .orderBy("year", "month") \
  .show(50)                 


+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2008|    1|    1|
|2008|    2|    1|
|2009|    5|    1|
|2009|   11|    1|
|2010|   11|    1|
|2011|    5|    1|
|2011|    9|    1|
|2011|   10|   11|
|2012|    2|    1|
|2012|   11|    1|
|2012|   12|    1|
|2013|    3|    1|
|2013|    8|    1|
|2013|    9|    2|
|2013|   10|    3|
|2013|   11|    2|
|2013|   12|    2|
|2014|    1|    2|
|2014|    2|    2|
|2014|    4|    2|
|2014|    6|    1|
|2014|    7|    1|
|2014|    8|    1|
|2014|    9|    1|
|2014|   10|    4|
|2014|   11|    4|
|2014|   12|    6|
|2015|    1|    1|
|2015|    2|    4|
|2015|    3|    5|
|2015|    4|    5|
|2015|    5|    6|
|2015|    6|    6|
|2015|    7|    7|
|2015|    8|    2|
|2015|    9|    7|
|2015|   10|   14|
|2015|   11|    4|
|2015|   12|   21|
|2016|    1|   43|
|2016|    2|   15|
|2016|    3|   17|
|2016|    4|   22|
|2016|    5|   13|
|2016|    6|   18|
|2016|    7|   28|
|2016|    8|   34|
|2016|    9|   48|
|2016|   10|   51|
|2016|   11|

In [0]:
df.select("date_added").show(20, truncate=False)

+------------------+
|date_added        |
+------------------+
|September 25, 2021|
|September 24, 2021|
|September 24, 2021|
|September 24, 2021|
|September 24, 2021|
|September 24, 2021|
|September 24, 2021|
|September 24, 2021|
|September 24, 2021|
|September 24, 2021|
|September 24, 2021|
|September 23, 2021|
|September 23, 2021|
|September 22, 2021|
|September 22, 2021|
|September 22, 2021|
|September 22, 2021|
|September 22, 2021|
|September 22, 2021|
|September 22, 2021|
+------------------+
only showing top 20 rows


9. Average duration of Tv Shows in seasons

In [0]:
from pyspark.sql.functions import col, split, avg, round

tv_df = df.filter(col("type") == "TV Show")

tv_df = tv_df.withColumn(
    "seasons",
    split(col("duration"), " ")[0].cast("int")
)

tv_df.select(
    round(avg("seasons"), 2).alias("average_seasons")
).show()


+---------------+
|average_seasons|
+---------------+
|           1.76|
+---------------+



Visual example

Original duration	After split	After [0]	Final seasons

2 Seasons	        ["2", "Seasons"]	    "2"	2
3 Seasons	        ["3", "Seasons"]	    "3"	3
1 Season	        ["1", "Season"]	        "1"	1

In [0]:
from pyspark.sql.functions import col, split, avg, round
result = (
    df
    .filter(col("type") == "TV Show")
    .filter(col("duration").like("%Season%"))
    .withColumn(
        "seasons",
        split(col("duration"), " ")[0].cast("int")
    )
    .agg(
        round(avg("seasons"), 2).alias("average_seasons")
    )
)
result.show()

+---------------+
|average_seasons|
+---------------+
|           1.76|
+---------------+



10.It's Christmas! What movies/TV shows are you binge-watching this weekend?

In [0]:
from pyspark.sql.functions import col, lower

christmas_df = df.filter(
    lower(col("title")).contains("christmas") |
    lower(col("description")).contains("christmas") |
    lower(col("listed_in")).contains("christmas")
)

christmas_df.select(
    "title", "type", "release_year", "listed_in"
).show(50, truncate=False)

+--------------------------------------------+-------+------------+------------------------------------------------------------+
|title                                       |type   |release_year|listed_in                                                   |
+--------------------------------------------+-------+------------+------------------------------------------------------------+
|Love Is a Story                             |Movie  |2015        |International Movies, Music & Musicals, Romantic Movies     |
|Home for Christmas                          |TV Show|2020        |International TV Shows, Romantic TV Shows, TV Comedies      |
|An Unremarkable Christmas                   |Movie  |2020        |Comedies, Dramas, International Movies                      |
|How To Ruin Christmas                       |TV Show|2020        |International TV Shows, TV Comedies, TV Dramas              |
|A California Christmas                      |Movie  |2020        |Comedies, Romantic Movies     

11. My friend is a Ryan Reynolds fan. What movies should I include in a movie compilation?

In [0]:
from pyspark.sql.functions import col, lower

ryan_movies = df.filter(
    (col("type") == "Movie") &
    (lower(col("cast")).contains("ryan reynolds"))
)

ryan_movies.select(
    "title", "release_year", "duration", "listed_in"
).show(50, truncate=False)

+-------------------------------------+------------+--------+------------------------------------+
|title                                |release_year|duration|listed_in                           |
+-------------------------------------+------------+--------+------------------------------------+
|Safe House                           |2012        |115 min |Action & Adventure                  |
|Green Lantern                        |2011        |114 min |Action & Adventure, Sci-Fi & Fantasy|
|Betty White: First Lady of Television|2018        |56 min  |Documentaries                       |
|6 Underground                        |2019        |129 min |Action & Adventure, Dramas          |
|Selfless                             |2015        |117 min |Sci-Fi & Fantasy, Thrillers         |
|Harold & Kumar Go to White Castle    |2004        |88 min  |Comedies, Cult Movies               |
|Just Friends                         |2005        |94 min  |Comedies, Romantic Movies           |
|Mississip

12. How many movies or shows were released during COVID season?

In [0]:
from pyspark.sql.functions import col

covid_df = df.filter(
    (col("release_year") >= 2020) &
    (col("release_year") <= 2021)
)

covid_df.groupBy("type") \
        .count() \
        .show()

+-------+-----+
|   type|count|
+-------+-----+
|  Movie|  794|
|TV Show|  751|
+-------+-----+



13. I have a short flight of 2 hours. Which movies or shows should I download?

In [0]:
from pyspark.sql.functions import col, split

flight_df = df.filter(col("type") == "Movie")

flight_df = flight_df.withColumn(
    "duration_minutes",
    split(col("duration"), " ")[0].cast("int")
)

flight_df.filter(col("duration_minutes") <= 120) \
         .select("title", "duration_minutes", "listed_in", "release_year") \
         .orderBy("duration_minutes") \
         .show(50, truncate=False)



+----------------------------------------------------------------------------+----------------+--------------------------------------------------------+------------+
|title                                                                       |duration_minutes|listed_in                                               |release_year|
+----------------------------------------------------------------------------+----------------+--------------------------------------------------------+------------+
|Silent                                                                      |3               |Children & Family Movies, Sci-Fi & Fantasy              |2014        |
|Sol Levante                                                                 |5               |Action & Adventure, Anime Features, International Movies|2020        |
|Cops and Robbers                                                            |8               |Dramas                                                  |2020        |
|Can